## Creando una Clase Base Iterativa Final

Este notebook contiene las clases finales del Capítulo 3.6 (lecciones 10, 11, 12, 13, 14 y 15), esto para no recargar tanto el notebook anterior que contiene las primeras lecciones eel capitulo.  
A continuación una breve descripción de la lección.  

En esta lección se agrega un nuevo método para cerrar la posición. Lo priemro es seleccionar la posición que se quiere cerrar, generalmente esta será la última barra en la que se operó o se realizó un trade.  

La mayor parte de este método lo que hace es crear impresiones (seis funciones de impresión). En resumen el método hace lo siguiente:  

Lo primero es obtener la fecha , precio y diferencial y luego se procede a cerrar la posición la cual puede ser corta o larga.  
Por el momento se asume que la posición final es larga, por lo tanto parqa cerrarla vendemos todas las unidades existentes. Así, self.units*price es positiva lo cual resulta en un precio final positivo. Y realmente, al vender una posición larga entonces nuestro balance actual se incrementa por los ingresos de venta.

Pero este código también funciona con una posición final corta, de tal manera que en este caso self.units es negativa y en consecuencia los ingresos por ventas finales son negativos, de tal manera que que tenemos que pagar dinero de nuestro balance actual para cerrar una posición corta; esto se suma un número negativo al balance actual y esto reduce el balance.
el próximo paso será establecer la posición como neutra, es decir, hacer self.units igual a cero.
Tener en cuenta que cerrar la posición final corresponde a un trade, entonces hay que incrementar los trades en uno.

A continuación se calcula el rendimiento de la estrategia, que es igual al balance actual menos el balance inicial que corresponde al dinero que se gana o se pierde con esta estrategia, lo dividimos entre el balance inicial y lo multiplicamos por 100, es decir se calcula el porcentaje de ganacias o pérdidas y también se imprime el balance actual


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

In [4]:
class IterativeBase():

    def __init__(self, symbol, start, end, amount):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_balance = amount
        self.current_balance = amount
        self.units = 0
        self.trades = 0 
        self.get_data()

    def get_data(self):
        raw = pd.read_csv("detailed.csv", parse_dates = ["time"], index_col = "time").dropna()
        raw = raw.loc[self.start:self.end]
        raw["returns"] = np.log(raw.price / raw.price.shift(1))
        self.data = raw

    def plot_data(self, cols = None):  
        if cols is None:
            cols = "price"
        self.data[cols].plot(figsize = (12, 8), title = self.symbol)
    
    def get_values(self, bar):
        date = str(self.data.index[bar].date())
        price = round(self.data.price.iloc[bar], 5)
        spread = round(self.data.spread.iloc[bar], 5)
        return date, price, spread
    
    def print_current_balance(self, bar):
        date, price, spread = self.get_values(bar)
        print("{} | Current Balance: {}".format(date, round(self.current_balance, 2)))
        
    def buy_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance -= units * price # reduce cash balance by "purchase price"
        self.units += units
        self.trades += 1
        print("{} |  Buying {} for {}".format(date, units, round(price, 5)))
    
    def sell_instrument(self, bar, units = None, amount = None):
        date, price, spread = self.get_values(bar)
        if amount is not None: # use units if units are passed, otherwise calculate units
            units = int(amount / price)
        self.current_balance += units * price # increases cash balance by "purchase price"
        self.units -= units
        self.trades += 1
        print("{} |  Selling {} for {}".format(date, units, round(price, 5)))
    
    def print_current_position_value(self, bar):
        date, price, spread = self.get_values(bar)
        cpv = self.units * price
        print("{} |  Current Position Value = {}".format(date, round(cpv, 2)))
    
    def print_current_nav(self, bar):
        date, price, spread = self.get_values(bar)
        nav = self.current_balance + self.units * price
        print("{} |  Net Asset Value = {}".format(date, round(nav, 2)))

    def close_position(self, bar):
        date, price, spread = self.get_values(bar)
        print(75 * "-")
        print("{} |+++ CLOSING FINAL POSITION +++".format(date))
        self.current_balance += self.units * price
        print("{} |closing position of {} for {}".format(date, self.units, price))
        self.units = 0
        self.trades += 1
        perf = (self.current_balance - self.initial_balance) / self.initial_balance * 100
        self.print_current_balance(bar)
        print("{} | net performance (%) = {}".format(date, round(perf, 2)))
        print("{} | number of trades = {}".format(date, self.trades))
        print(75 * "-")
    

In [5]:
bc = IterativeBase("EURUSD", "2006-12-31", "2020-06-30", 100000)

In [6]:
bc.buy_instrument(0, amount=100000)

2006-12-31 |  Buying 75766 for 1.31985


In [7]:
bc.print_current_balance(0)

2006-12-31 | Current Balance: 0.24


In [8]:
bc.print_current_position_value(0)

2006-12-31 |  Current Position Value = 99999.76


In [10]:
bc.print_current_balance(-1)

2020-06-29 | Current Balance: 0.24


In [11]:
bc.print_current_position_value(-1)

2020-06-29 |  Current Position Value = 85108.71


In [12]:
bc.print_current_nav(-1)

2020-06-29 |  Net Asset Value = 85108.95


In [13]:
bc.close_position(-1)

---------------------------------------------------------------------------
2020-06-29 |+++ CLOSING FINAL POSITION +++
2020-06-29 |closing position of 75766 for 1.12331
2020-06-29 | Current Balance: 85108.95
2020-06-29 | net performance (%) = -14.89
2020-06-29 | number of trades = 2
---------------------------------------------------------------------------


In [14]:
bc.data

,price,spread,returns
time,,,
2006-12-31 22:00:00+00:00,1.31985,0.00100,NaN
2007-01-01 22:00:00+00:00,1.32734,0.00015,0.005659
2007-01-02 22:00:00+00:00,1.31688,0.00015,-0.007912
2007-01-03 22:00:00+00:00,1.30845,0.00015,-0.006422
2007-01-04 22:00:00+00:00,1.30025,0.00100,-0.006287
...,...,...,...
2020-06-23 21:00:00+00:00,1.12507,0.00030,-0.005151
2020-06-24 21:00:00+00:00,1.12180,0.00023,-0.002911
2020-06-25 21:00:00+00:00,1.12184,0.00041,0.000036


In [15]:
bc.data.price[-1] / bc.data.price[0] - 1

C:\Users\marioL\AppData\Local\Temp\ipykernel_8936\3677939590.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  bc.data.price[-1] / bc.data.price[0] - 1


np.float64(-0.1489108610826988)